In [ ]:
# Does 38's NFT attractor survive all-to-all ZZ coupling? One seed, ~30-40 min on a T4.

!pip install -q qiskit==1.4.6 qiskit-aer-gpu==0.15.1 qiskit-algorithms==0.4.0 qiskit-optimization==0.7.0 qiskit-ibm-runtime==0.29.0 2>&1 | tail -5

import time, numpy as np
from qiskit.circuit.library import TwoLocal, PauliTwoDesign
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeCairoV2
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import NFT
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

def p(*a):
    print(*a, flush=True)

n_qubits = 15
rng_h = np.random.default_rng(0)
terms = []
for i in range(n_qubits):
    lab = ["I"] * n_qubits; lab[i] = "Z"
    terms.append(("".join(lab), float(rng_h.uniform(-2, 2))))
# all-to-all ZZ this time, the one thing changed from notebook 38
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        lab = ["I"] * n_qubits; lab[i] = "Z"; lab[j] = "Z"
        # *0.3 keeps the energy scale near 38's, with 105 pair terms instead of 14
        terms.append(("".join(lab), float(rng_h.uniform(-2, 2)) * 0.3))
ising_op = SparsePauliOp.from_list(terms)
p(f"Hamiltonian: {len(terms)} terms (15 single-Z + {n_qubits*(n_qubits-1)//2} pairwise-ZZ, all-to-all)")

cairo_nm = NoiseModel.from_backend(FakeCairoV2())
sampler = AerSampler(
    seed=42,
    options={"backend_options": {
        "noise_model": cairo_nm, "method": "statevector", "device": "GPU",
        "batched_shots_gpu": False,
        "max_parallel_threads": 0, "max_parallel_experiments": 0,
    }},
)
sampler.options.default_shots = 2000

ansatz_configs = {
    "PauliTwo":          lambda: PauliTwoDesign(num_qubits=n_qubits, reps=3, seed=42),
    "TwoLocal linear":   lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="linear", reps=3),
    "TwoLocal circular": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="circular", reps=3),
    "TwoLocal pairwise": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="pairwise", reps=3),
    "TwoLocal full (control)": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="full", reps=3),
}

results = {}
seed = 42
for name, factory in ansatz_configs.items():
    ansatz = factory()
    ansatz_d = transpile(ansatz.decompose(reps=10), backend=sampler._backend, optimization_level=1)
    init = np.random.default_rng(seed).uniform(-np.pi, np.pi, size=ansatz_d.num_parameters)
    t0 = time.time()
    vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=NFT(maxiter=40), initial_point=init)
    result = vqe.compute_minimum_eigenvalue(ising_op)

    bound = ansatz_d.assign_parameters(list(result.optimal_parameters.values())
                                        if isinstance(result.optimal_parameters, dict)
                                        else list(result.optimal_parameters))
    bound.measure_all()
    job = sampler.run([transpile(bound, optimization_level=1)], shots=2000)
    counts = job.result()[0].data.meas.get_counts()
    best_bitstring = max(counts, key=counts.get)

    elapsed = time.time() - t0
    results[name] = (float(np.real(result.eigenvalue)), best_bitstring, elapsed)
    p(f"  {name:26s}: energy={float(np.real(result.eigenvalue)):9.4f}  bits={best_bitstring}  ({elapsed:.0f}s)")

p(f"\n--- pairwise Hamming distances, all-to-all coupling ---")
names = list(ansatz_configs.keys())
def hd(a, b):
    return sum(x != y for x, y in zip(a, b))
for i, n1 in enumerate(names):
    for n2 in names[i+1:]:
        d = hd(results[n1][1], results[n2][1])
        p(f"  {n1:26s} vs {n2:26s}: {d}")

cluster = ["PauliTwo", "TwoLocal linear", "TwoLocal circular", "TwoLocal pairwise"]
cluster_dists = [hd(results[a][1], results[b][1]) for i,a in enumerate(cluster) for b in cluster[i+1:]]
control_dists = [hd(results[c][1], results["TwoLocal full (control)"][1]) for c in cluster]
p(f"\nmax distance within the sparse-entanglement group: {max(cluster_dists)}")
p(f"min distance from the control to that group: {min(control_dists)}")
p("(notebook 38, nearest-neighbour coupling: within-group 0-1, control 3-5)")